# 06. Reading the shape of terrain

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/06-terrain.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=06-terrain.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/06-terrain.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `36841a084960204c2a97d8cb0ac3641a1f4d3b8b7247c847462730d95fccce1a`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 6: Reading the shape of terrain; abstract contour illustration](assets/06-terrain.svg)

# 06. Reading the shape of terrain

Slope, hillshade, contours, and 3D

**Learning goals:** Derive terrain products with TerrIO; compare shading with elevation; inspect an interactive surface.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=06-terrain.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Relief and its derivatives

Slope describes a local rate of change. Hillshade describes illumination under an assumed light, so dark pixels need not be low. Both horizontal and vertical units are metres here. Real DEMs need a vertical datum and often an explicit scale conversion.

In [ ]:
dem = raster(); z = dem.read(1)
slope = terrio.dem(dem, "slope").read(1)
shade = terrio.dem(dem, "hillshade", azimuth=315, altitude=45).read(1)
image_grid([z], ["Synthetic elevation"]); plt.show()
image_grid([slope], ["Slope"], ["magma"], units="degrees"); plt.show()
image_grid([shade], ["Hillshade: northwest illumination"], ["gray"], units="brightness (0–255)"); plt.show()

## A transparent numerical comparison

NumPy gradients provide a separate finite-difference approximation. TerrIO's DEM algorithm can differ at edges and in its neighbourhood stencil. Compare the interior and report the difference rather than claiming equivalence from visual similarity.

In [ ]:
dy, dx = np.gradient(z.astype(float), 30, 30)
estimate = np.rad2deg(np.arctan(np.hypot(dx, dy)))
interior = (slice(2, -2), slice(2, -2))
delta = slope[interior] - estimate[interior]
print("Interior mean absolute difference (degrees):", float(np.abs(delta).mean()))
assert np.all((slope[interior] >= 0) & (slope[interior] <= 90))
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(delta.ravel(), bins=30)
ax.set(xlabel="TerrIO − NumPy slope (degrees)", ylabel="Pixel count")
plt.show()

## Contours are derived vector features

A contour joins interpolated equal-elevation positions. Its interval controls how much detail is shown. The derived lines inherit the source uncertainty; more lines do not imply better measurements.

In [ ]:
contours = terrio.contour(dem, interval=50, with_z=True, field_name="elevation")
terrio.write_vector(contours, str(OUTPUT / "contours.gpkg"))
print("Contour features:", len(contours), "fields:", contours.fields)
assert len(contours) > 0

## Rotate the landscape

Drag to rotate; scroll to zoom. The surface uses projected metre coordinates. A vertical exaggeration of 2 helps expose structure and is explicitly disclosed in the title. The colour bar continues to show unmodified elevations.

In [ ]:
interactive(surface(z, exaggeration=2))

## Your investigation

Change the hillshade azimuth to 135°. Identify features whose apparent relief reverses. Compare the 3D surface at 1× and 4× exaggeration.

### Discussion guide

Illumination changes apparent relief without changing elevation. Exaggeration changes the display aspect ratio, not the reported height values.

## Sources and credit

- [Source 1](https://terrio.dev/python/)
- [Source 2](https://terrio.dev/cli/#dem)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.